# 1. Check result file and answer counts

In [2]:
import pandas as pd
import numpy as np
import os
output_dir = '../data/outputs/temp_1_qwen/'

for model_name in os.listdir(output_dir):
    if model_name.endswith('.json'):
        data = pd.read_json(os.path.join(output_dir, model_name))
        responses = data['output']
        counts = {k: 0 for k in ['text_basic', 'text_cot', 'text_instructive']}
        for prompt_format in ['text_basic', 'text_cot', 'text_instructive']:
            for response in responses:
                text = response[prompt_format]
                if text != 'SKIPPED':
                    counts[prompt_format] += 1
        # if sum(counts.values()) == 1500:
        #     print("'" + model_name.replace('.json', '') + "'", end = ' ')
        # print('\n' * 20)
        print(f'Model: {model_name}, responses count: {counts}')
        # for question_type in np.unique([md['question_type'] for md in data['meta_data']]):
        #     # print an example per question type
        #     for i, row in data.iterrows():
        #         if row['meta_data']['question_type'] == question_type:
        #             print(f'Example for question type: {question_type}')
        #             # print('Prompt:')
        #             # print(row['input'])
        #             print('Responses:')
        #             for prompt_format in ['text_basic', 'text_cot', 'text_instructive']:
        #                 print(f'  {prompt_format}:')
        #                 print(f'    {row["output"][prompt_format]}')
        #             break
        # print('\n' * 20)


Model: Llama-3.1-8B-Instruct.json, responses count: {'text_basic': 500, 'text_cot': 500, 'text_instructive': 500}
Model: Llama-3.3-70B-Instruct.json, responses count: {'text_basic': 500, 'text_cot': 500, 'text_instructive': 500}
Model: Mistral-7B-Instruct-v0.3.json, responses count: {'text_basic': 500, 'text_cot': 500, 'text_instructive': 500}
Model: Mistral-Small-24B-Instruct-2501.json, responses count: {'text_basic': 500, 'text_cot': 500, 'text_instructive': 500}
Model: Mixtral-8x7B-Instruct-v0.1.json, responses count: {'text_basic': 500, 'text_cot': 500, 'text_instructive': 500}
Model: OLMo-2-1124-13B-Instruct.json, responses count: {'text_basic': 500, 'text_cot': 500, 'text_instructive': 500}
Model: OLMo-2-1124-7B-Instruct.json, responses count: {'text_basic': 500, 'text_cot': 500, 'text_instructive': 500}
Model: Qwen2.5-32B-Instruct.json, responses count: {'text_basic': 500, 'text_cot': 500, 'text_instructive': 500}
Model: Qwen2.5-72B-Instruct.json, responses count: {'text_basic':

# 2. Aggregate evaluation results

In [3]:
import os 
import json 
from tqdm import tqdm 
import re 
import pandas as pd
import numpy as np

runid_list = [
    'temp_1_qwen',
    'temp_1_gpt41'
]

# result_path = '../data/evaluations/temp_1_qwen/'
# model_lst = [f.replace('.json', '') for f in os.listdir(result_path) if '.json' in f]
model_lst = [
    'Mistral-7B-Instruct-v0.3',
    'claude-3-7-sonnet-20250219',
    'gpt-4.1-mini-2025-04-14',
    'claude-3-haiku-20240307',
    'gpt-4.1-2025-04-14',
    'Llama-3.1-8B-Instruct',
    'OLMo-2-1124-13B-Instruct',
    'Llama-3.3-70B-Instruct',
    'Mistral-Small-24B-Instruct-2501',
    'Qwen2.5-7B-Instruct',
    'Qwen2.5-72B-Instruct',
    'Qwen2.5-32B-Instruct',
    'gemini-2.0-flash',
    'Mixtral-8x7B-Instruct-v0.1',
    'deepseek-reasoner',
    'OLMo-2-1124-7B-Instruct',
    'deepseek-chat'
]

In [4]:
traits = ['Fluency', 'Flexibility', 'Originality', 'Elaboration']

tmp_eval_output = 'Fluency: 5  \nFlexibility: 5  \nOriginality: 4  \nElaboration: 5  \n'
tmp_eval_pairwise = 'Fluency: B  \nFlexibility: A  \nOriginality: A  \nElaboration: A  \n'

def extract_scores(eval_output, pairwise=False):
    if not pairwise:
        pattern = r'(\w+): (\d+)'
        matches = re.findall(pattern, eval_output.split('### Scores ###')[1] if '### Scores ###' in eval_output else eval_output)
        scores = {match[0]: int(match[1]) for match in matches}
        return scores
    else:
        scores = {}
        for trait in traits:
            if trait + ':' not in eval_output:
                print(f"Warning: {trait} not found in eval_output")
            else:

                pattern = f'{trait}: ([A,B])'
                matches = re.findall(pattern, eval_output.split('### Judgement ###')[1] if '### Judgement ###' in eval_output else eval_output)
                # print(matches)
                scores[trait] = matches[0] if matches else None
        return scores
extract_scores(tmp_eval_output)

{'Fluency': 5, 'Flexibility': 5, 'Originality': 4, 'Elaboration': 5}

In [8]:
traits = ['Fluency', 'Flexibility', 'Originality', 'Elaboration']
all_ttct_outputs = {}
for model in tqdm(model_lst):
    all_ttct_outputs[model] = {'raw_data': {}, 'score_avg': {trait: [] for trait in traits}, 'score_std': {trait: [] for trait in traits}}
    tmp_results = {}
    for runid in runid_list:
        result_path = f'../data/evaluations/{runid}/'
        if model + '.json' in os.listdir(result_path):
            with open(os.path.join(result_path, model + '.json'), 'r') as f:
                tmp_results[runid] = json.load(f)
        else:
            print(f"No results found for {model}, runid: {runid}")
    for runid, data in tmp_results.items():
        for item in data:
            item['evaluation_scores'] = {
                'text_cot': extract_scores(item['evaluation']['text_cot'])
            }
        
        all_ttct_outputs[model]['raw_data'][runid] = data
        for trait in traits:
            trait_scores = [
                item['evaluation_scores']['text_cot'][trait] 
                for item in data if trait in item['evaluation_scores']['text_cot'] 
                and item['evaluation_scores']['text_cot'][trait] is not None
            ]
            all_ttct_outputs[model]['score_avg'][trait].append(round(np.mean(trait_scores) / 5, 4) if trait_scores else None)
            all_ttct_outputs[model]['score_std'][trait].append(round(np.std(trait_scores) / 5, 4) if trait_scores else None)
    
    for trait in traits:
        all_ttct_outputs[model]['score_avg'][trait] = np.nanmean(all_ttct_outputs[model]['score_avg'][trait])
        all_ttct_outputs[model]['score_std'][trait] = np.nanmean(all_ttct_outputs[model]['score_std'][trait])
        # break

100%|██████████| 17/17 [00:02<00:00,  6.81it/s]


In [9]:
tmp_results.keys()

dict_keys(['temp_1_qwen', 'temp_1_gpt41'])

In [10]:
# [all_ttct_outputs[model]['score_avg'] for model in all_ttct_outputs]

In [11]:
all_ttct_outputs.keys()

dict_keys(['Mistral-7B-Instruct-v0.3', 'claude-3-7-sonnet-20250219', 'gpt-4.1-mini-2025-04-14', 'claude-3-haiku-20240307', 'gpt-4.1-2025-04-14', 'Llama-3.1-8B-Instruct', 'OLMo-2-1124-13B-Instruct', 'Llama-3.3-70B-Instruct', 'Mistral-Small-24B-Instruct-2501', 'Qwen2.5-7B-Instruct', 'Qwen2.5-72B-Instruct', 'Qwen2.5-32B-Instruct', 'gemini-2.0-flash', 'Mixtral-8x7B-Instruct-v0.1', 'deepseek-reasoner', 'OLMo-2-1124-7B-Instruct', 'deepseek-chat'])

In [12]:
model_order = [
    'Mistral-7B-Instruct-v0.3',
    'Qwen2.5-7B-Instruct',
    'OLMo-2-1124-7B-Instruct',
    'Llama-3.1-8B-Instruct',
    'OLMo-2-1124-13B-Instruct',
    'Mistral-Small-24B-Instruct-2501',
    'Qwen2.5-32B-Instruct',
    'Mixtral-8x7B-Instruct-v0.1',
    'Llama-3.3-70B-Instruct',
    'Qwen2.5-72B-Instruct',
    'gpt-4.1-2025-04-14',
    'gpt-4.1-mini-2025-04-14',
    'gemini-2.0-flash',
    'deepseek-reasoner',
    'deepseek-chat',
    'claude-3-7-sonnet-20250219',
    'claude-3-haiku-20240307',
]

all_score_df = []
for model in model_order:
    if model in all_ttct_outputs:
        tmp_dict = {
            'model': model,
        }
        for trait in traits:
            tmp_dict[trait + '_avg'] = all_ttct_outputs[model]['score_avg'][trait]
            tmp_dict[trait + '_std'] = all_ttct_outputs[model]['score_std'][trait]
        all_score_df.append(tmp_dict)

In [14]:
all_score_df = pd.DataFrame(all_score_df)
all_score_df

,model,Fluency_avg,Fluency_std,Flexibility_avg,Flexibility_std,Originality_avg,Originality_std,Elaboration_avg,Elaboration_std
0,Mistral-7B-Instruct-v0.3,0.72680,0.16500,0.82700,0.17350,0.59330,0.17590,0.57215,0.11910
1,Qwen2.5-7B-Instruct,0.76585,0.14740,0.86975,0.13650,0.61105,0.15455,0.62935,0.13090
2,OLMo-2-1124-7B-Instruct,0.62865,0.18860,0.68940,0.22090,0.50975,0.18060,0.53105,0.13905
3,Llama-3.1-8B-Instruct,0.73400,0.17115,0.80410,0.17335,0.59355,0.16530,0.59940,0.13300
4,OLMo-2-1124-13B-Instruct,0.58140,0.21215,0.62505,0.24775,0.48565,0.18620,0.50695,0.15455
5,Mistral-Small-24B-Instruct-2501,0.70350,0.17125,0.79035,0.19310,0.56870,0.16590,0.57250,0.12165
6,Qwen2.5-32B-Instruct,0.74480,0.15240,0.85635,0.13385,0.61240,0.14960,0.63330,0.10970
7,Mixtral-8x7B-Instruct-v0.1,0.70370,0.18270,0.76655,0.20135,0.55220,0.17965,0.51855,0.13665
8,Llama-3.3-70B-Instruct,0.72470,0.16480,0.81795,0.16970,0.59720,0.16925,0.60770,0.11585
9,Qwen2.5-72B-Instruct,0.76700,0.15100,0.89290,0.13120,0.62915,0.13835,0.62905,0.10145


In [15]:
all_score_df.to_csv('../data/evaluations/evaluation_summary_qwen_gpt41.csv', index=False)